Bing Chen 

ST 554 HW9 

4/10/2026

# Read in data 
I chose Breast Cancer Wisconsin Data from UCI machine learning repository. This dataset contains information about characteristics of the cell nuclei of breast mass and whether the patient was diagnosed with malignant or benign tumor.

First, intall the repo and read in the data. 

In [1]:
#!pip install ucimlrepo

In [2]:
#from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
#breast_cancer_wisconsin_diagnostic = fetch_ucirepo(id=17) 
  
# see variable information 
#print(breast_cancer_wisconsin_diagnostic.variables) 


These are the target and feature variables: 

In [4]:
#X = breast_cancer_wisconsin_diagnostic.data.features 
#y = breast_cancer_wisconsin_diagnostic.data.targets 

#print(X.head(), y.head())

UCI website was down so I read it the downloaded data into pandas dataframe. Note that this data have no headers. So we'll manually add them. 

In [5]:
import pandas as pd 

breast_cancer_wisconsin_diagnostic = pd.read_csv("wdbc.data", header = None)

In [6]:
columns = [
    "id", "Diagnosis",
    "radius1","texture1","perimeter1","area1","smoothness1","compactness1","concavity1","concave_points1","symmetry1","fractal_dimension1",
    "radius2","texture2","perimeter2","area2","smoothness2","compactness2","concavity2","concave_points2","symmetry2","fractal_dimension2",
    "radius3","texture3","perimeter3","area3","smoothness3","compactness3","concavity3","concave_points3","symmetry3","fractal_dimension3"
]


breast_cancer_wisconsin_diagnostic.columns = columns

breast_cancer_wisconsin_diagnostic.head()

,id,Diagnosis,radius1,texture1,perimeter1,area1,smoothness1,compactness1,concavity1,concave_points1,...,radius3,texture3,perimeter3,area3,smoothness3,compactness3,concavity3,concave_points3,symmetry3,fractal_dimension3
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


The target varible we want to make prediction on is `Diagnosis`, whether the tumor is M (malignant) or B (benign). So let's create the response and explanatory variables. 

In [7]:
# target
y = breast_cancer_wisconsin_diagnostic["Diagnosis"]

# predictors
X = breast_cancer_wisconsin_diagnostic.drop(columns=["Diagnosis", "id"])

# Prepare the data

Let's now read in all the modules we'll need and start a `pyspark` session. 


In [8]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import SQLTransformer, StringIndexer, VectorAssembler, StandardScaler, PCA
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

spark = SparkSession.builder \
    .appName("SparkDataCheck") \
    .config("spark.sql.ansi.enabled", "false") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 23:05:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


We need to transform out data into a spark SQL data frame. To do that, we'll first combine the feature and target variables into a `pandas` data frame, then into spark SQL dataframe.

In [9]:
df = pd.concat([X, y], axis=1)
cancer = spark.createDataFrame(df)


After we done that, we can split the data into training and test sets. 

In [10]:
train, test = cancer.randomSplit([0.8,0.2], seed = 1)
print(train.count(), test.count())

26/04/10 23:05:38 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


457 112


Let's check the count within each class. 

In [11]:
cancer.groupBy("Diagnosis").count().show()

+---------+-----+
|Diagnosis|count|
+---------+-----+
|        M|  212|
|        B|  357|
+---------+-----+



We will use accuracy to compare the models. Accuracy measures the overall proportion of correctly classified observations. Be cautious that, here, the dataset contains 212 malignant and 357 benign cases with indicates a light class imbalance.

Since our target variable is binary, I chose to fit models: 
1. LogisticRegression：estimates the probability of a binary outcome using a logistic function. 
2. DecisionTreeClassifier: splits the data into branches based on feature values to make predictions. It creates a tree-like structure of decision rules. 
3. RandomForestClassifier: An ensemble method that builds many decision trees on different subsets of the data and averages their predictions.

# Fit the Model 
To fit the model, we'll put the predictors into a single column called 'features'. Then define the transformations we want to do and the evaluator. 

Since for `MLlib`, the label must be numeric. We can use `StringIndexer` to convert it into binary 0 and 1's; combine all feature columns into a single feature vector via `Vector Assembler`; standardize features using `StandardScaler`, reduce dimensionality using `PCA`! 

In [12]:
feature_cols = [c for c in cancer.columns if c not in ["Diagnosis", "id"]]

indexer = StringIndexer(inputCol="Diagnosis", outputCol="label")

assembler = VectorAssembler(inputCols=feature_cols, outputCol="assembled_features")

scaler = StandardScaler(
    inputCol="assembled_features",
    outputCol="scaled_features"
)

pca = PCA(
    k=10,  
    inputCol="scaled_features",
    outputCol="features"
)

# Accuracy evaluator
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)




Now let's fit the logistic regression. We'll build a pipeline to fit the model. Then, use cross validation over the grid of tuning parameters to choose the best logistic regression model. 

In [13]:
lr = LogisticRegression(labelCol="label", featuresCol="features")

lr_pipeline = Pipeline(stages=[indexer, assembler, scaler, pca, lr])

lr_param_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.0, 0.01, 0.1])
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
    .build()
)

lr_cv = CrossValidator(
    estimator=lr_pipeline,
    estimatorParamMaps=lr_param_grid,
    evaluator=evaluator,
    numFolds=5,
    seed=12
)

lr_cv_model = lr_cv.fit(train)
lr_predictions = lr_cv_model.transform(test)
lr_accuracy = evaluator.evaluate(lr_predictions)

print("Logistic Regression Accuracy:", lr_accuracy)


Logistic Regression Accuracy: 0.9553571428571429


Great, now we move on to fitting a decision tree with the same procedures! 

In [15]:

dt = DecisionTreeClassifier(labelCol="label", featuresCol="features")

dt_pipeline = Pipeline(stages=[indexer, assembler, scaler, pca, dt])

dt_param_grid = (
    ParamGridBuilder()
    .addGrid(dt.maxDepth, [3, 5, 10])
    .addGrid(dt.minInstancesPerNode, [1, 2, 5])
    .build()
)

dt_cv = CrossValidator(
    estimator=dt_pipeline,
    estimatorParamMaps=dt_param_grid,
    evaluator=evaluator,
    numFolds=5,
    seed=12
)

dt_cv_model = dt_cv.fit(train)
dt_predictions = dt_cv_model.transform(test)
dt_accuracy = evaluator.evaluate(dt_predictions)

print("Decision Tree Accuracy:", dt_accuracy)

Decision Tree Accuracy: 0.9107142857142857


Random Forest. 

In [16]:
rf = RandomForestClassifier(labelCol="label", featuresCol="features")

rf_pipeline = Pipeline(stages=[indexer, assembler, scaler, pca, rf])

rf_param_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [20, 50, 100])
    .addGrid(rf.maxDepth, [3, 5, 10])
    .build()
)

rf_cv = CrossValidator(
    estimator=rf_pipeline,
    estimatorParamMaps=rf_param_grid,
    evaluator=evaluator,
    numFolds=5,
    seed=1234
)

rf_cv_model = rf_cv.fit(train)
rf_predictions = rf_cv_model.transform(test)
rf_accuracy = evaluator.evaluate(rf_predictions)

print("Random Forest Accuracy:", rf_accuracy)

Random Forest Accuracy: 0.9107142857142857


Lastly, we want to evaluate the best models from each class to find the best model! 

In [17]:
print("Logistic Regression: ", lr_accuracy, "\n",
      "Decision Tree: ", dt_accuracy, "\n",
      "Random Forest: ", rf_accuracy)


Logistic Regression:  0.9553571428571429 
 Decision Tree:  0.9107142857142857 
 Random Forest:  0.9107142857142857


Looks like logistic regression gives the best accuracy! Let's check which model is chosen as the best:

In [25]:
my_list = []
for i in range(len(lr_param_grid)):
    my_list.append([lr_cv_model.avgMetrics[i], lr_param_grid[i].values()])
my_list

[[0.9757570462001258, dict_values([0.0, 0.0])],
 [0.9757570462001258, dict_values([0.0, 0.5])],
 [0.9757570462001258, dict_values([0.0, 1.0])],
 [0.9737467329044429, dict_values([0.01, 0.0])],
 [0.9719381700408242, dict_values([0.01, 0.5])],
 [0.9762636318130944, dict_values([0.01, 1.0])],
 [0.934757357141652, dict_values([0.1, 0.0])],
 [0.9214059940206025, dict_values([0.1, 0.5])],
 [0.9104355832805053, dict_values([0.1, 1.0])]]

We can also find it via: 

In [24]:
best_model = lr_cv_model.bestModel
best_lr = best_model.stages[-1]
print("Best regParam:", best_lr._java_obj.getRegParam())
print("Best elasticNetParam:", best_lr._java_obj.getElasticNetParam())

Best regParam: 0.01
Best elasticNetParam: 1.0


CV selected a regularization parameter of 0.01 and an elastic net parameter of 1.0 for the logistic regression model. 